# Beyond GDP: Human Development and Capability Analysis

## Notebook 6 — Outlier Detection

An outlier is a country that stands far apart from the rest, like one student
scoring 99 when most score around 65. We find these unusual countries using the
z-score, which measures how far a value is from the average.

Finding them matters because they are often the most interesting cases, and because
extreme values can pull the averages around.


### What is an outlier?

An outlier is a country that stands far apart from all the rest. Think of a class
where most students score 60 to 70 marks, but one scores 99 and another scores 5.
Those two are outliers, because they are far from the crowd.

Finding outliers matters for two reasons:
1. They are often the most interesting stories (like a very rich but tiny country).
2. They can pull our results around (one extreme value can shift the average), so we
   should know which countries they are.

### How do we find them?

We use a simple number called the z-score. The z-score tells how far a value is from
the average:
- z-score near 0 means the country is close to average, normal.
- z-score above +3 means the country is far above average (for example, very high GDP).
- z-score below -3 means the country is far below average.

The common rule: if any of a country's numbers has a z-score above 3 (or below -3),
we treat that country as an outlier, because it sits far from the crowd. For each
country we look at its most extreme number, and flag the ones where that goes past 3.

    

## 1. Setup

We load the cleaned master dataset. This notebook works on the same data as before.

In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

master_path = Path(r"D:\beyond-gdp-capability-analysis\data\processed\master_dataset.csv")
df = pd.read_csv(master_path)

print("Shape:", df.shape)
df.head(3)

Shape: (130, 18)


,country,code,hdi,life_expectancy,expected_schooling,mean_schooling,gni_per_capita,happiness_score,social_support_contrib,freedom_contrib,generosity_contrib,corruption_contrib,gdp_per_capita,health_expenditure,unemployment,inflation,population,cpi_score
0,Iceland,ISL,0.972,82.691,18.85059,13.908926,69116.93736,7.525,1.617,0.819,0.258,0.182,82138.789297,8.705101,3.518,8.736303,385663.0,77
1,Norway,NOR,0.970,83.308,18.79285,13.117962,112710.02110,7.302,1.517,0.835,0.224,0.484,87497.217965,9.429255,3.574,5.517850,5519601.0,81
2,Switzerland,CHE,0.970,83.954,16.66753,13.949121,81948.90177,7.060,1.425,0.759,0.173,0.498,100623.549627,11.690961,4.043,2.135401,8888822.0,80


## 2. Find outliers using the z-score

The z-score tells how far a value is from the average:
- near 0 means close to average (normal),
- above +3 means far above average,
- below -3 means far below average.

For each country we take its most extreme number across the main measures. If that
goes past 3, we treat the country as an outlier, meaning it sits far from the crowd.

In [12]:
check_cols = ["hdi", "gdp_per_capita", "happiness_score", "life_expectancy",
              "health_expenditure", "unemployment", "inflation", "cpi_score", "population"]

z_scores = df[check_cols].apply(lambda x: (x - x.mean()) / x.std())

df["max_z"] = z_scores.abs().max(axis=1)

outliers = df[df["max_z"] > 3].sort_values("max_z", ascending=False)

print("Number of outlier countries:", len(outliers))
print()
print(outliers[["country", "max_z"]].to_string(index=False))

Number of outlier countries: 11

      country    max_z
      Lebanon 8.981321
        India 7.641064
        China 7.489658
    Argentina 5.226683
 South Africa 5.138773
   Luxembourg 4.448017
United States 3.485929
      Ireland 3.414028
     Botswana 3.399148
  Afghanistan 3.357906
  Switzerland 3.171486


In [13]:
def which_extreme(row):
    z_row = ((df[check_cols] - df[check_cols].mean()) / df[check_cols].std()).loc[row.name]
    col = z_row.abs().idxmax()
    return f"{col} (z={z_row[col]:.1f})"

outliers["reason"] = outliers.apply(which_extreme, axis=1)

print(outliers[["country", "max_z", "reason"]].to_string(index=False))

      country    max_z                     reason
      Lebanon 8.981321          inflation (z=9.0)
        India 7.641064         population (z=7.6)
        China 7.489658         population (z=7.5)
    Argentina 5.226683          inflation (z=5.2)
 South Africa 5.138773       unemployment (z=5.1)
   Luxembourg 4.448017     gdp_per_capita (z=4.4)
United States 3.485929 health_expenditure (z=3.5)
      Ireland 3.414028     gdp_per_capita (z=3.4)
     Botswana 3.399148       unemployment (z=3.4)
  Afghanistan 3.357906   happiness_score (z=-3.4)
  Switzerland 3.171486     gdp_per_capita (z=3.2)


**What the outliers tell us:**

11 countries stand far from the crowd. Each one is unusual for a clear, real reason,
not by mistake:

Economy gone wrong (extreme inflation):
- Lebanon (z=9.0) and Argentina (z=5.2) had very high inflation. Both went through
  serious money crises, so this is real, not an error.

Very large population:
- India (z=7.6) and China (z=7.5) are outliers simply because they are the two most
  populous countries in the world. Their population dwarfs everyone else.

High unemployment:
- South Africa (z=5.1) and Botswana (z=3.4) have unusually high unemployment, a
  long-known problem in both.

Very high income or spending:
- Luxembourg (z=4.4), Ireland (z=3.4) and Switzerland (z=3.2) stand out on GDP per
  capita, they are among the richest. The United States (z=3.5) stands out on health
  spending, it spends far more on health than any other country.

Very low well-being:
- Afghanistan (z=-3.4) is an outlier for the lowest happiness score in the data,
  which fits its conflict and hardship.

The key point: none of these are data errors. Each is a genuine extreme, an economic
crisis, a huge population, a rich economy, or a country in deep difficulty. Knowing
they exist also explains why some averages get pulled around, and why measures like
inflation and unemployment had weak links with HDI earlier, a few extreme countries
were stretching them.